[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C09_Reasoning_TTC_Course/04_search_mcts/04_search_mcts.ipynb)

# 04 · 推理即搜索：beam / Tree-of-Thoughts / MCTS 从零

<span style="background:#1f6feb;color:#fff;padding:2px 8px;border-radius:4px;font-size:12px">CPU</span>

把多步推理形式化为树搜索（状态 = 部分推理、动作 = 下一步、价值 = 能否到达正确答案），在一个**可精确判对**的玩具环境里，公平地比较"线性采样"与"树搜索"两大类 test-time compute 策略。

**本 notebook 你将完成：**

1. 搭建玩具推理环境 **算术迷宫**：固定步数内用 +/−/× 把起点数变到目标值（状态可枚举、对错可精确判定），并用动态规划求出 oracle 价值 $V^*$；
2. 构造合成 **proposer**（带偏好噪声的 policy：好动作概率更高但不保证）与 **value model**（带噪声的 $\hat V$）；
3. 在**同等节点预算**下实现并对比四种策略：greedy / 独立采样（any-correct）/ beam search（价值模型打分）/ **从零实现 MCTS**（UCT 选择、展开、随机 rollout、回传）；
4. 画出"成功率 vs 节点预算"四线对比图、"价值噪声 $\sigma_v$ → 各策略相对优势"相图、UCT 探索系数 $c$ 扫描；
5. 完成 **4 道 ✏️ 练习**：`uct_score` / `beam_step` / `mcts_backpropagate` / `compare_strategies`。

参考文献：Yao et al. 2023 (ToT, arXiv:2305.10601) · Hao et al. 2023 (RAP, arXiv:2305.14992) · Kocsis & Szepesvári 2006 (UCT) · Snell et al. 2024 (arXiv:2408.03314) · DeepSeek-R1 (arXiv:2501.12948)


## 1 · 玩具推理环境：算术迷宫

模拟多步推理需要一个"理想实验台"：**每条推理路径可精确判对、整个状态空间可枚举**（从而能算出 oracle 价值 $V^*$ 当作上帝视角的对照）。我们用一个 24 点式的算术搜索任务：

- **状态** $s=(v, d)$：当前数值 $v$、已走步数 $d$ —— 对应"部分推理轨迹"；
- **动作**：从固定算子集 `OPS`（+2, +5, −3, ×2, ×3, −1）中选一个施加到 $v$ —— 对应"下一个推理步"；
- **成功**：恰好走 $D=6$ 步后 $v = $ target —— 对应"最终答案正确"（稀疏奖励，只在终点）；
- **oracle 价值** $V^*(v,d)\in\{0,1\}$：从 $(v,d)$ 出发还能否到达 target。倒序动态规划即可精确求出。

题目按构造必可解（target 由随机走 $D$ 步生成），所以失败全部归因于**策略**而非任务。分支因子 $K=6$，完整树有 $6^6 = 46656$ 条路径——足够大到暴力枚举"不公平"，足够小到 oracle 可算。两个 ×  算子让可达值快速发散，"仍有救"的状态随深度变稀疏——解是稀疏的，瞎走很难撞上。


In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)

# ---- 环境定义：算术迷宫 ------------------------------------------------
OPS = [("+", 2), ("+", 5), ("-", 3), ("*", 2), ("*", 3), ("-", 1)]
K, D = len(OPS), 6                      # 分支因子 K=6，固定深度 D=6

def apply_op(v, op):
    sym, k = op
    if sym == "+": return v + k
    if sym == "-": return v - k
    return v * k

def make_instance(rng):
    """随机生成一道题 (起点 s0, 目标 target)；target 由随机走 D 步得到 => 必可解。"""
    s0 = int(rng.integers(1, 10))
    v = s0
    for _ in range(D):
        v = apply_op(v, OPS[int(rng.integers(K))])
    return (s0, v)

def oracle_good_sets(s0, target):
    """动态规划求精确价值 V*：good[d] = 深度 d 处所有"仍能到达 target"的数值集合。
    先正向枚举每层可达集 layers[d]，再从终点倒推。"""
    layers = [{s0}]
    for _ in range(D):
        layers.append({apply_op(v, op) for v in layers[-1] for op in OPS})
    good = [set() for _ in range(D + 1)]
    good[D] = {target} if target in layers[D] else set()
    for d in range(D - 1, -1, -1):
        good[d] = {v for v in layers[d]
                   if any(apply_op(v, op) in good[d + 1] for op in OPS)}
    return good

N_INST = 24
instances = [make_instance(rng) for _ in range(N_INST)]
oracles   = [oracle_good_sets(s0, t) for s0, t in instances]

s0, t0 = instances[0]
g0 = oracles[0]
print(f"示例题目：从 {s0} 出发，恰走 {D} 步（每步 ∈ {[s + str(k) for s, k in OPS]}）到达 {t0}")
layers = [{s0}]
for _ in range(D):
    layers.append({apply_op(v, op) for v in layers[-1] for op in OPS})
print("各深度可达状态数  :", [len(L) for L in layers])
print("各深度'仍有救'状态数:", [len(g0[d]) for d in range(D + 1)])
print("根节点 V* =", 1.0 if s0 in g0[0] else 0.0, "（按构造必为 1）")


## 2 · 合成 proposer 与 value model

真实系统里 proposer 是 LLM 的步骤生成、value model 是 03 章的 PRM。这里用**围绕 oracle 加噪声**的合成版本，从而可以拧两个旋钮做对照实验：

- **policy（proposer）**：$\;p(a\mid s)\propto \exp\big(\beta\, V^*(s')+\sigma_p\,\varepsilon\big)$ —— $\beta$ 控制"会做题的程度"，$\sigma_p$ 控制偏好噪声。好动作概率更高，但不保证被选中；
- **value model**：$\;\hat V(s)=\mathrm{clip}\big(V^*(s)+\sigma_v\,\varepsilon,\,0,\,1\big)$ —— $\sigma_v=0$ 即 oracle，$\sigma_v$ 越大越接近"瞎猜的 PRM"。

噪声按 (状态, 动作) **缓存**：同一状态反复查询答案一致——它表现得像一个固定但不完美的模型，而非每次重掷骰子。这保证 greedy 是确定性的、搜索算法的剪枝错误是"系统性"的（更贴近真实 PRM 的系统性偏差）。


In [ ]:
class Models:
    """合成的提议模型 + 价值模型（围绕 oracle V* 加缓存噪声）。"""
    def __init__(self, good, beta=1.5, sigma_p=1.5, sigma_v=0.3, seed=0):
        self.good, self.beta = good, beta
        self.sigma_p, self.sigma_v = sigma_p, sigma_v
        self.rng = np.random.default_rng(seed)
        self._pol, self._val = {}, {}

    def _vstar(self, v, d):
        return 1.0 if v in self.good[d] else 0.0

    def policy_probs(self, v, d):
        """返回 K 个动作的概率：softmax(beta * V*(child) + sigma_p * eps)。"""
        key = (v, d)
        if key not in self._pol:
            logits = np.array([self.beta * self._vstar(apply_op(v, op), d + 1)
                               for op in OPS])
            logits = logits + self.sigma_p * self.rng.standard_normal(K)
            e = np.exp(logits - logits.max())
            self._pol[key] = e / e.sum()
        return self._pol[key]

    def value(self, v, d):
        """带噪价值估计 V̂(v,d) ∈ [0,1]；sigma_v=0 即 oracle。"""
        key = (v, d)
        if key not in self._val:
            noisy = self._vstar(v, d) + self.sigma_v * self.rng.standard_normal()
            self._val[key] = float(np.clip(noisy, 0.0, 1.0))
        return self._val[key]

# sanity check：找一个"部分动作好、部分动作坏"的状态，验证 policy 确实偏向好动作
demo = None
for d in range(D):
    for v in sorted(g0[d]):
        n_good = sum(apply_op(v, op) in g0[d + 1] for op in OPS)
        if 0 < n_good <= K // 2:
            demo = (v, d, n_good); break
    if demo: break
v_demo, d_demo, n_good = demo
m0 = Models(g0, seed=1)
p = m0.policy_probs(v_demo, d_demo)
good_idx = [i for i, op in enumerate(OPS) if apply_op(v_demo, op) in g0[d_demo + 1]]
print(f"演示状态 (v={v_demo}, d={d_demo})：{n_good}/{K} 个动作'仍有救'")
print(f"policy 给好动作的概率和 = {p[good_idx].sum():.3f}（无偏好基线 = {n_good/K:.3f}）")
print(f"oracle 价值（σ_v=0）: V̂ = {Models(g0, sigma_v=0.0, seed=1).value(v_demo, d_demo)}")
print(f"带噪价值（σ_v=0.3）: V̂ = {m0.value(v_demo, d_demo):.3f}")


## 3 · 四种策略与算力会计

统一记账单位：**生成一个新状态（节点）计 1**。同一预算 $B$ 下：

| 策略 | 形态 | 预算换算 |
|---|---|---|
| greedy | 宽度 1，每步取 policy argmax | 恒为 $D$，不随 $B$ 变化 |
| 独立采样 | $N$ 条完整轨迹，any-correct（02 章基线） | $N = B/D$ |
| beam search | 每层 $W\!\cdot\!K$ 个候选、价值模型留 top-$W$ | $W = B/(D\,K)$ |
| MCTS | 迭代 = 展开 1 节点 + rollout 剩余深度 | 迭代到累计节点数 $\ge B$ |

MCTS 每次迭代四阶段（selection → expansion → simulation → backpropagation），selection 用 **UCT**（Kocsis & Szepesvári 2006）：

$$\mathrm{UCT}(s,a) \;=\; \underbrace{\frac{W(s,a)}{N(s,a)}}_{\text{exploitation}} \;+\; c\sqrt{\frac{\ln N(s)}{N(s,a)}}\,,\qquad N(s,a)=0 \Rightarrow +\infty$$

simulation 的回报用 AlphaGo 式混合：$r=(1-\lambda)\,r_{\text{rollout}}+\lambda\,\hat V(\text{leaf})$ —— 价值模型由此成为搜索启发（03 章 PRM 的接口）。

成功判据默认为 **any-correct**：预算内"碰到过"一条完整且正确的路径就算赢——这隐含假设终点可被免费精确验证（本环境查 `v == target` 即可，类似代码任务跑单测）。第 5 节会引入更苛刻的"**提交制**"判据（无终点验证、必须交一条路径）作对照——你会看到**评测判据本身会改变哪些超参数重要**。


In [ ]:
def run_greedy(inst, good, models):
    """宽度 1：每步取 policy 概率最大的动作。开销恒为 D 个节点。"""
    s0, target = inst
    v = s0
    for d in range(D):
        v = apply_op(v, OPS[int(np.argmax(models.policy_probs(v, d)))])
    return v == target, D

def run_sampling(inst, good, models, budget, rng):
    """独立采样 N = budget // D 条完整轨迹，any-correct。"""
    s0, target = inst
    used, success = 0, False
    for _ in range(max(1, budget // D)):
        v = s0
        for d in range(D):
            v = apply_op(v, OPS[int(rng.choice(K, p=models.policy_probs(v, d)))])
            used += 1
        success = success or (v == target)
    return success, used

def run_beam(inst, good, models, budget):
    """beam search：宽度 W = budget // (D*K)；每层展开全部孩子、按价值模型留 top-W。"""
    s0, target = inst
    W = max(1, budget // (D * K))
    beams, used = [s0], 0
    for d in range(D):
        cands = [apply_op(v, op) for v in beams for op in OPS]
        used += len(cands)
        order = np.argsort([-models.value(c, d + 1) for c in cands])
        beams = [cands[i] for i in order[:W]]
    return target in beams, used

ok, n = run_beam(instances[0], g0, Models(g0, seed=42), budget=120)
print(f"beam search 示例：题目 0，预算 120 → 宽度 {max(1, 120 // (D * K))}，"
      f"实际用 {n} 节点，成功 = {ok}")


In [ ]:
class Node:
    """MCTS 树节点：v=数值, d=深度, N=访问次数, W=累计回报, Q = W/N。"""
    __slots__ = ("v", "d", "parent", "children", "untried", "N", "W")
    def __init__(self, v, d, parent=None):
        self.v, self.d, self.parent = v, d, parent
        self.children, self.untried = [], list(range(K))
        self.N, self.W = 0, 0.0

def uct(parent, child, c):
    if child.N == 0:
        return float("inf")                       # 未访问的孩子无条件优先
    return child.W / child.N + c * math.sqrt(math.log(parent.N) / child.N)

def run_mcts(inst, good, models, budget, rng, c=1.4, lam=0.5):
    """从零 MCTS：UCT 选择 → 展开 → 随机 rollout（与 V̂ 按 λ 混合）→ 回传。
    返回 (any-correct 成功, 用掉节点数, 树根)——树根供"提交制"判据复用。"""
    s0, target = inst
    root = Node(s0, 0)
    used, success = 0, False
    while used < budget:
        # ① selection：沿 UCT 最大孩子下行到未完全展开/终止节点
        node = root
        while node.d < D and not node.untried and node.children:
            node = max(node.children, key=lambda ch: uct(node, ch, c))
        # ② expansion：随机选一个未尝试动作，生成新孩子
        if node.d < D and node.untried:
            i = node.untried.pop(int(rng.integers(len(node.untried))))
            child = Node(apply_op(node.v, OPS[i]), node.d + 1, parent=node)
            node.children.append(child)
            node = child
        used += 1                                  # 展开/重访叶节点计 1（防预算空转）
        success = success or (node.d == D and node.v == target)
        # ③ simulation：用 policy 随机 rollout 到深度 D
        v, d = node.v, node.d
        while d < D:
            v = apply_op(v, OPS[int(rng.choice(K, p=models.policy_probs(v, d)))])
            d, used = d + 1, used + 1
        success = success or (v == target)
        reward = (1 - lam) * (1.0 if v == target else 0.0) + lam * models.value(node.v, node.d)
        # ④ backpropagation：沿路径自下而上更新 N / W
        while node is not None:
            node.N, node.W = node.N + 1, node.W + reward
            node = node.parent
    return success, used, root

# —— 同一预算下四种策略的成功率（24 道题 × n_rep 次重复取均值）——
def evaluate(strategy, budget, sigma_v=0.3, n_rep=2, **kw):
    wins = tot = 0
    for rep in range(n_rep):
        for j, (inst, good) in enumerate(zip(instances, oracles)):
            models = Models(good, sigma_v=sigma_v, seed=100 + 1000 * rep + j)  # 各策略共用同一组模型
            r = np.random.default_rng(200 + 1000 * rep + j)
            if   strategy == "greedy":   ok, _ = run_greedy(inst, good, models)
            elif strategy == "sampling": ok, _ = run_sampling(inst, good, models, budget, r)
            elif strategy == "beam":     ok, _ = run_beam(inst, good, models, budget)
            elif strategy == "mcts":     ok, _, _ = run_mcts(inst, good, models, budget, r, **kw)
            wins += ok; tot += 1
    return wins / tot

B = 120
print(f"节点预算 B = {B}，价值噪声 σ_v = 0.3，题目数 = {N_INST}\n")
for s in ["greedy", "sampling", "beam", "mcts"]:
    print(f"  {s:>8}: 成功率 = {evaluate(s, B):.3f}")


In [ ]:
# —— 成功率 vs 节点预算：四线对比 ——————————————————————
budgets = [30, 60, 120, 240, 480]
strategies = ["greedy", "sampling", "beam", "mcts"]
curves = {s: [evaluate(s, B) for B in budgets] for s in strategies}

plt.figure(figsize=(7, 4.2))
for s, mk in zip(strategies, "osD^"):
    plt.plot(budgets, curves[s], marker=mk, label=s)
plt.xscale("log")
plt.xlabel("node budget B (log)"); plt.ylabel("success rate (any-correct)")
plt.title(f"success vs budget (sigma_v=0.3, {N_INST} instances x 2 reps)")
plt.ylim(-0.03, 1.05); plt.grid(alpha=0.3); plt.legend(); plt.tight_layout(); plt.show()

for s in strategies:
    print(f"{s:>8}:", [round(x, 2) for x in curves[s]])
print("\n解读：greedy 是水平线（不花预算就不涨）；采样靠 1-(1-p)^N 缓慢爬升；")
print("MCTS 把预算导向高价值子树，同预算稳压采样；beam 在 σ_v=0.3（价值模型很准）时")
print("宽度 1 就接近天花板——好的 PRM 能把搜索需求压缩到极小，但它对 σ_v 极其脆弱（见下图）。")


In [ ]:
# —— 相图：价值噪声 σ_v ↑ 时搜索的退化 + UCT 探索系数 c 扫描 ——————
sigmas = [0.0, 0.5, 1.0, 1.5, 2.0, 3.0]
B = 120
base = evaluate("sampling", B)            # 采样不用价值模型，与 σ_v 无关
beam_curve = [evaluate("beam", B, sigma_v=s) for s in sigmas]
mcts_curve = [evaluate("mcts", B, sigma_v=s) for s in sigmas]

# c 扫描：同时用两种成功判据
#   any-correct：预算内碰到正确终点即赢（终点有免费验证器）
#   提交制     ：搜索结束后必须提交"访问数最多"的一条路径（无终点验证）
def mcts_submit(inst, good, models, budget, rng, c):
    """提交制：跑完 MCTS 后沿 max-N 孩子下行提交一条路径，无孩子处用 policy 贪心补全。"""
    _, _, root = run_mcts(inst, good, models, budget, rng, c=c)
    node = root
    while node.d < D:
        if node.children:
            node = max(node.children, key=lambda ch: ch.N)
        else:
            v = apply_op(node.v, OPS[int(np.argmax(models.policy_probs(node.v, node.d)))])
            node = Node(v, node.d + 1)
    return node.v == inst[1]

def eval_c(c, budget=240, n_rep=4):
    w_any = w_sub = tot = 0
    for rep in range(n_rep):
        for j, (inst, good) in enumerate(zip(instances, oracles)):
            m1 = Models(good, seed=100 + 1000 * rep + j)
            r1 = np.random.default_rng(200 + 1000 * rep + j)
            ok, _, _ = run_mcts(inst, good, m1, budget, r1, c=c)
            m2 = Models(good, seed=100 + 1000 * rep + j)
            r2 = np.random.default_rng(200 + 1000 * rep + j)
            w_any += ok; w_sub += mcts_submit(inst, good, m2, budget, r2, c); tot += 1
    return w_any / tot, w_sub / tot

cs = [0.0, 0.2, 0.5, 1.0, 2.0, 4.0, 8.0]
c_any, c_sub = zip(*[eval_c(c) for c in cs])

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ax = axes[0]
ax.plot(sigmas, beam_curve, marker="D", label="beam")
ax.plot(sigmas, mcts_curve, marker="^", label="mcts (lam=0.5)")
ax.axhline(base, ls="--", color="gray", label=f"sampling baseline ({base:.2f})")
ax.set_xlabel("value-model noise sigma_v"); ax.set_ylabel("success rate (any-correct)")
ax.set_title("value quality -> search advantage"); ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(cs, c_any, marker="^", label="any-correct")
ax.plot(cs, c_sub, marker="s", label="submit most-visited path")
ax.set_xlabel("UCT exploration constant c"); ax.set_ylabel("success rate")
ax.set_title("MCTS c sweep (B=240, sigma_v=0.3)"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f"相图：σ_v=0 时 beam={beam_curve[0]:.2f} / mcts={mcts_curve[0]:.2f}，均高于 sampling={base:.2f}；")
print(f"      σ_v={sigmas[-1]} 时 beam={beam_curve[-1]:.2f} —— 硬剪枝 + 无回溯，跌破暴力采样基线；")
print(f"      mcts 仅缓慢回落到 {mcts_curve[-1]:.2f}：rollout 的真实回报（λ=0.5）+ 免费终点验证给了它下界。")
print(f"      若无免费验证器（如开放数学题），MCTS 的退化会陡得多——见右图提交制曲线整体低于 any-correct。")
print(f"c 扫描（any-correct）: {dict(zip(cs, [round(a, 2) for a in c_any]))}")
print(f"c 扫描（提交制）     : {dict(zip(cs, [round(a, 2) for a in c_sub]))}")
print("any-correct 下 c 几乎不重要（碰上就赢，探索方式是二阶效应）；提交制下呈倒 U——")
print("c=0 纯剥削被早期噪声锁死，c 过大退化为均匀轮询，访问统计不再集中于好路径。")
print("教训：评测判据本身决定了哪些超参数重要——换个 metric，结论就换了。")


---
## ✏️ 练习 1：实现 `uct_score(parent_N, child, c)`

实现 UCT 打分：`child` 是形如 `{"N": int, "W": float}` 的 dict，`parent_N` 是父节点访问次数。规则：

- `child["N"] == 0` 时返回 `float("inf")`（未访问的孩子无条件优先）；
- 否则返回 $\dfrac{W}{N} + c\sqrt{\dfrac{\ln(\text{parent\_N})}{N}}$。

**提示**：3–5 行；用 `math.log` / `math.sqrt`；注意 $c=0$ 时应退化为纯 $Q=W/N$ 排序（贪心）。边界：不需要处理 `parent_N == 0`（回传保证父先于子被计数）。


In [ ]:
def uct_score(parent_N, child, c):
    # TODO: child["N"] == 0 时返回 float("inf")（未访问优先）
    # TODO: 否则返回 exploitation 项 W/N + exploration 项 c*sqrt(ln(parent_N)/N)
    raise NotImplementedError


In [ ]:
# —— 练习 1 自测 ——
unvisited = {"N": 0, "W": 0.0}
a = {"N": 10, "W": 8.0}    # Q=0.8，访问多
b = {"N": 2,  "W": 1.0}    # Q=0.5，访问少
assert uct_score(12, unvisited, 1.4) == float("inf")            # 未访问子节点优先
assert abs(uct_score(12, a, 0.0) - 0.8) < 1e-9                  # c=0 退化为纯 Q（贪心）
assert uct_score(12, a, 0.0) > uct_score(12, b, 0.0)
assert uct_score(12, b, 5.0) > uct_score(12, a, 5.0)            # c 大时偏向少访问者
assert abs(uct_score(12, b, 1.4) - (0.5 + 1.4 * math.sqrt(math.log(12) / 2))) < 1e-9
print("✅ 练习 1 通过")


## ✏️ 练习 2：实现 `beam_step(beams, width, value_fn, d)`

实现 beam search 的**单层推进**：`beams` 是深度 `d` 的一组数值状态，把每个状态用全部 `OPS` 展开为候选孩子（深度 `d+1`），用 `value_fn(child, d+1)` 打分，返回**按分数从高到低排序后的前 `width` 个**候选（list）。

**提示**：3–5 行；列表推导生成 `len(beams)*K` 个候选 → `sort(key=..., reverse=True)` → 切片 `[:width]`。边界：候选不足 `width` 个时全部返回；不要求去重。


In [ ]:
def beam_step(beams, width, value_fn, d):
    # TODO: 1) 把每个 beam 状态用全部 OPS 展开成候选孩子（深度 d+1）
    # TODO: 2) 按 value_fn(child, d+1) 从高到低排序
    # TODO: 3) 返回前 width 个候选（list）
    raise NotImplementedError


In [ ]:
# —— 练习 2 自测 ——
oracle_v = lambda v, d: 1.0 if v in g0[d] else 0.0      # 题目 0 的 oracle 价值
out = beam_step([s0], 3, oracle_v, 0)
assert len(out) == 3
assert all(any(c == apply_op(s0, op) for op in OPS) for c in out)   # 都是合法孩子
sc = [oracle_v(c, 1) for c in out]
assert sc == sorted(sc, reverse=True)                                # 按分数降序
assert oracle_v(out[0], 1) == 1.0                                    # oracle 下榜首必"仍有救"
assert len(beam_step([s0, s0], 50, oracle_v, 0)) == min(50, 2 * K)   # 候选不足时全收
print("✅ 练习 2 通过")


## ✏️ 练习 3：实现 `mcts_backpropagate(path, reward)`

实现 MCTS 第④阶段：`path` 是从根到叶的节点列表，每个节点是 `{"N": int, "W": float}`。对路径上**每个**节点执行 `N += 1`、`W += reward`（$Q = W/N$ 由两者隐式决定，不单独存），**原地修改**并 `return path`。

**提示**：3 行循环即可。边界：`reward` 可以是 0/1 之外的混合值（λ 混合后是小数）；路径长度可以为 1（只有根）。


In [ ]:
def mcts_backpropagate(path, reward):
    # TODO: 对 path 中每个节点：node["N"] += 1, node["W"] += reward
    # TODO: 原地修改，最后 return path
    raise NotImplementedError


In [ ]:
# —— 练习 3 自测 ——
path = [{"N": 0, "W": 0.0} for _ in range(3)]
mcts_backpropagate(path, 1.0)
mcts_backpropagate(path, 0.0)
mcts_backpropagate(path, 1.0)
assert all(n["N"] == 3 for n in path)                          # 每个节点被计数 3 次
assert all(abs(n["W"] - 2.0) < 1e-9 for n in path)             # 累计回报 = 1+0+1
assert all(abs(n["W"] / n["N"] - 2 / 3) < 1e-9 for n in path)  # Q = W/N
leaf = [{"N": 5, "W": 2.0}]
assert mcts_backpropagate(leaf, 0.5) is leaf                   # 原地修改并返回
assert leaf[0]["N"] == 6 and abs(leaf[0]["W"] - 2.5) < 1e-9
print("✅ 练习 3 通过")


## ✏️ 练习 4：实现 `compare_strategies(budget)` —— 同预算公平比较协议

把第 9 节（讲解页）的公平比较协议封装成函数：在 **oracle 价值函数（$\sigma_v=0$）** 下，用同一节点预算 `budget` 跑全部四种策略，返回 `{"greedy": .., "sampling": .., "beam": .., "mcts": ..}`（值为成功率）。

**提示**：1–3 行——直接复用上面的 `evaluate(strategy, budget, sigma_v=0.0)`。检查点：oracle 价值下 beam 应稳定打满（剪枝永不剪错），MCTS 也应 ≥ 独立采样——如果不是，说明你的预算换算或 σ_v 传参有误。


In [ ]:
def compare_strategies(budget):
    # TODO: 用 sigma_v=0.0（oracle 价值函数）对四种策略各调一次 evaluate，
    #       返回 {"greedy": .., "sampling": .., "beam": .., "mcts": ..}
    raise NotImplementedError


In [ ]:
# —— 练习 4 自测 ——
res = compare_strategies(120)
assert set(res) == {"greedy", "sampling", "beam", "mcts"}       # 四种策略齐全
assert all(0.0 <= v <= 1.0 for v in res.values())
assert res["beam"] >= res["sampling"] - 1e-9                    # oracle 价值下搜索 ≥ 暴力采样
assert res["mcts"] >= res["sampling"] - 1e-9
print({k: round(v, 3) for k, v in res.items()})
print("✅ 练习 4 通过")


---
## 📖 参考答案


In [ ]:
# 练习 1 参考答案（先自己做，再对照）
def uct_score(parent_N, child, c):
    if child["N"] == 0:
        return float("inf")
    return child["W"] / child["N"] + c * math.sqrt(math.log(parent_N) / child["N"])


In [ ]:
# 练习 2 参考答案（先自己做，再对照）
def beam_step(beams, width, value_fn, d):
    cands = [apply_op(v, op) for v in beams for op in OPS]
    cands.sort(key=lambda c: value_fn(c, d + 1), reverse=True)
    return cands[:width]


In [ ]:
# 练习 3 参考答案（先自己做，再对照）
def mcts_backpropagate(path, reward):
    for node in path:
        node["N"] += 1
        node["W"] += reward
    return path


In [ ]:
# 练习 4 参考答案（先自己做，再对照）
def compare_strategies(budget):
    return {s: evaluate(s, budget, sigma_v=0.0)
            for s in ["greedy", "sampling", "beam", "mcts"]}


---
## 小结

- **推理即搜索**：状态 = 部分推理、动作 = 下一步、价值 = 能否到达正确答案。树搜索相对线性采样多了两个能力：**中途剪枝**与**回溯**，其全部收益来自"中间状态可被判断"。
- **四种策略一条谱**：greedy（宽 1、无纠错）→ 独立采样（轨迹级 best-of-N）→ beam（层级剪枝、无回溯）→ MCTS（UCT 非均匀分配 + 回溯）。预算换算 $N=B/D$、$W=B/(DK)$ 是公平比较的前提。
- **UCT** $= Q + c\sqrt{\ln N(s)/N(s,a)}$：未访问优先、$c=0$ 纯剥削、$c\to\infty$ 均匀轮询。$c$ 的重要性取决于**评测判据**：any-correct（有免费终点验证）下是二阶效应；提交制（交访问数最多的路径）下呈倒 U。
- **价值函数是杠杆**：$\sigma_v=0$ 时 beam/MCTS 同预算碾压采样；$\sigma_v$ 增大后 beam（硬剪枝、无回溯）**率先跌破暴力采样基线**；MCTS 靠 rollout 的无偏回报退化平缓——但去掉免费验证器后它同样大幅缩水。搜索放大判断的好坏，而暴力采样没有杠杆、永不背叛 baseline。
- 评测含义：比较 TTC 策略必须同预算（节点/token 两本账）、必含暴力采样与 oracle-价值两个对照、并写明成功判据——这正是 `compare_strategies` 的玩具版协议，模块 05 把它推广到 compute-optimal scaling。


---
## 🎯 真实数据胶囊题：真实奖励上的 UCB1/UCT 选择

MCTS 用 UCT 平衡探索与利用：`score = 均值 + c·sqrt(ln(N)/n_i)`。用真实 GSM8K 难度构造若干“动作臂”的真实奖励，实现 UCB1 选择，验证它最终更多选到高奖励臂、但也探索过其他臂。

> 本模块新增的**真实数据**练习：自包含、用真实 GSM8K 把本章方法跑一遍。先做 TODO，`assert` 全过即通关，文末有参考答案。

In [ ]:
import os, json, urllib.request, re
import numpy as np
CACHE=os.path.expanduser("~/.reasoning_ttc_data"); os.makedirs(CACHE,exist_ok=True)
def _f(url,fn):
    p=os.path.join(CACHE,fn)
    if not os.path.exists(p): urllib.request.urlretrieve(url,p)
    return p
def gsm8k(n=300):
    p=_f("https://raw.githubusercontent.com/openai/grade-school-math/master/grade_school_math/data/test.jsonl","gsm8k_test.jsonl")
    return [json.loads(l) for l in open(p).read().splitlines()[:n]]
def gold(a): return a.split("####")[-1].strip().replace(",","")
def steps(a): return max(1, a.count("<<"))   # 真实推理步数代理

rows=gsm8k(200)
ks=np.array([steps(r["answer"]) for r in rows])
rng=np.random.default_rng(0)
# 4 个"策略臂"，真实奖励 = 在某难度子集上的成功率
arms_reward=[0.3,0.5,0.7,0.45]   # 真实(隐藏)期望奖励
def pull(arm): return 1.0 if rng.random()<arms_reward[arm] else 0.0

**练习**：实现 `ucb1_select(values, counts, t, c)`：返回 UCB1 分数最高的臂索引。未被拉过的臂(count=0)应优先(分数视为无穷)。

In [ ]:
def ucb1_select(values, counts, t, c=1.4):
    # TODO: 对每臂 score = values[i] + c*sqrt(ln(t)/counts[i])；count=0 时 +inf；返回 argmax
    raise NotImplementedError


In [ ]:
# 自测：跑 2000 步 UCB，最优臂应被拉最多次
K=4; values=np.zeros(K); counts=np.zeros(K)
for t in range(1,2001):
    a=ucb1_select(values, counts, t)
    r=pull(a); counts[a]+=1; values[a]+=(r-values[a])/counts[a]
assert counts.argmax()==int(np.argmax(arms_reward)), "UCB 应最多拉真实最优臂"
assert (counts>0).all(), "UCB 应探索过所有臂"
print(f"UCB 各臂拉取次数={counts.astype(int)}, 最优臂(#{np.argmax(arms_reward)})被拉最多 ✓")


### 📖 参考答案

In [ ]:
def ucb1_select(values, counts, t, c=1.4):
    scores=np.where(counts==0, np.inf, values + c*np.sqrt(np.log(t)/np.maximum(counts,1)))
    return int(np.argmax(scores))
print("✓ UCT = UCB1 用在树搜索上，是 MCTS/AlphaGo 的探索核心")